In [1]:
import pandas as pd
import os
from unidecode import unidecode

In [2]:
def combinar_h_v(df):
    """
    Combina pares de columnas que empiezan por 'h' y 'v' en un DataFrame,
    creando nuevas columnas con los valores combinados. Además, devuelve
    una lista de las columnas creadas.

    Args:
        df (pd.DataFrame): DataFrame original.

    Returns:
        pd.DataFrame: DataFrame con columnas combinadas añadidas.
        list: Lista de nombres de las columnas creadas.
    """
    columnas_h = [col for col in df.columns if col.startswith('h')]
    columnas_v = [col for col in df.columns if col.startswith('v')]
    
    columnas_creadas = []  # Lista para almacenar nombres de las columnas creadas

    for col_h, col_v in zip(columnas_h, columnas_v):
        nueva_columna = f"{col_h}_{col_v}"  # Nombre de la nueva columna
        df[nueva_columna] = df[col_h].astype(str) + " " + df[col_v].astype(str)
        columnas_creadas.append(nueva_columna)  # Agregar el nombre de la columna a la lista
    
    return df, columnas_creadas

In [3]:
def formato_df(df):
    """
    Formatea un DataFrame con datos de calidad del aire, asegurando una estructura homogénea y bien organizada.

    Pasos del proceso:

    1. Convertir las columnas clave a tipo string: "provincia", "municipio", "estacion", "magnitud", "ano", "mes" y "dia".
    2. Asegurar que "mes" y "dia" tengan siempre dos dígitos.
    3. Crear la columna "fecha" combinando "ano", "mes" y "dia".
    4. Unificar las columnas de hora y validación con `combinar_h_v()`.
    5. Transformar las columnas de hora en filas mediante `pd.melt()`.
    6. Extraer el estado de validación del valor y ajustar la columna "hora".
    7. Modificar el formato de "hora", cambiando "24" por "23:59" y ajustando las demás horas.
    8. Crear la columna "fecha_hora_f" con formato datetime.
    9. Extraer el estado de validación de la medida desde la columna "valor".
    10. Corregir valores con comas decimales, reemplazando `,` por `.` en "valor".
    11. Convertir "valor" a tipo float.
    12. Generar un identificador único "id_medida" para cada observación.
    13. Construir el código "codigo_estacion" combinando "provincia", "municipio" y "estacion".
    14. Crear el código de técnica basado en "punto_muestreo".
    15. Eliminar filas con valores nulos o extremos.

    Args:
        df (pd.DataFrame): DataFrame con los datos de calidad del aire.

    Returns:
        pd.DataFrame: DataFrame formateado con la estructura correcta y listo para su análisis.
    """
    # Pasamos las columnas de "provincia", "municipio", "estacion" y "magnitud" a str porque no vamos a operar con esos números
    df[["provincia", "municipio", "estacion", "magnitud"]] = df[["provincia", "municipio", "estacion", "magnitud"]].astype(str)
    # Nos aseguramos de que las columnas de fecha sean str
    df[["ano", "mes", "dia"]] = df[["ano", "mes", "dia"]].astype(str)

    # Rellenamos mes y día para que siempre tengan 2 dígitos
    df["mes"] = df["mes"].str.zfill(2)
    df["dia"] = df["dia"].str.zfill(2)

    # Creamos la columna de fecha
    df["fecha"] = df["ano"] + "-" + df["mes"] + "-" + df["dia"]

    # Combinamos las columnas de hora y validación
    df, columnas_creadas = combinar_h_v(df)

    # Transformamos el DataFrame para que las columnas de hora queden en filas
    df = df.melt(
        id_vars=["provincia", "municipio", "estacion", "magnitud", "punto_muestreo", "fecha"],
        value_vars=columnas_creadas,
        var_name="hora",
        value_name="valor"
    )

    # Ahora separamos la letra que acompaña al valor para crear la columna de validación y dejamos solo la hora
    df["hora"] = df["hora"].str.split("_", expand=True)[0].str.extract("(\\d+)")[0]

    # Ajustamos el formato de la hora para casos especiales
    df["hora"] = df["hora"].apply(lambda x: '23:59' if x == '24' else f"{str(x)}:00")

    # Creamos una columna de fecha y hora con formato datetime
    df["fecha_hora_f"] = pd.to_datetime(df["fecha"].astype(str) + " " + df["hora"].astype(str))

    # Extraemos el estado de validación de la medida
    df["validacion"] = df["valor"].str.split(" ", expand=True)[1]
    df["valor"] = df["valor"].str.split(" ", expand=True)[0]

    # Corrige los valores con comas decimales antes de convertir a float
    df["valor"] = df["valor"].str.replace(",", ".")  # Reemplaza comas por puntos en decimales

    # Convertimos "valor" a float
    df["valor"] = df["valor"].astype(float)

    # Creamos un id único para cada medida
    df["id_medida"] = df["punto_muestreo"] + "_" + df["fecha"] + "_" + df["hora"] + "_" + df["validacion"]

    # Formamos el código de estación
    df["codigo_estacion"] = df["provincia"].astype(str).str.zfill(2) + \
                            df["municipio"].astype(str).str.zfill(3) + \
                            df["estacion"].astype(str).str.zfill(3)
    df["municipio"] = df["municipio"].astype(str).str.zfill(3)
    df["codigo_tecnica"] = df["punto_muestreo"].apply(lambda x: x.split('_')[-1])
    df["codigo_tecnica"] = df["codigo_tecnica"].str.replace("A", "47")
    df = df[df["valor"].abs() < 1000]
    df = df.dropna()
    df = df.drop_duplicates(subset=["id_medida"], inplace = True)
    
    return df

In [4]:
def unir_archivos(carpeta_entrada, patron_nombre, carpeta_salida, nombre_salida):
    """
    Une los archivos de una carpera según el nombre dado y los guarda como un único parquet en la carpeta de salida

    Parámetros:
        carpeta_entrada (str): ruta de la carpeta donde se encuentran los archivos que queremos unir.
        patron_nombre (str): patrón de nombre de los archivos a unir
        carpeta_salida (str): ruta de la carpeta donde se guardan los archivos unidos.
        nombre_salida (str): nombre del archivo de salida (sin extensión).

    Retorna:
        archivos: Lista con los nombres de los archivos unidos.
    """
    carpeta = carpeta_entrada
    archivos = [archivo for archivo in os.listdir(carpeta) if archivo.lower().startswith(patron_nombre.lower())]
    df_lista = [pd.read_csv(os.path.join(carpeta, archivo), sep=";", parse_dates = True, encoding="latin1", low_memory = False) for archivo in archivos]
    df_unido = pd.concat(df_lista, ignore_index=True)
    df_unido.to_parquet(f"{carpeta_salida}/{nombre_salida}.parquet", index=False)
    print(archivos)
    print(f"archivo {nombre_salida}.parquet creado en {carpeta_salida}")
    return df_unido

In [5]:
def crear_df_medidas(df_cmadrid, df_madrid, carpeta_salida):
    df_madrid = formato_df_madrid(df_madrid)
    if df_cmadrid.columns.equals(df_madrid.columns):
        df_medidas = pd.concat([df_madrid, df_cmadrid], ignore_index=True)
        df_medidas.to_parquet(f"{carpeta_salida}/medidas.parquet", index=False)
        print(f"Archivo medidas.parquet creado en {carpeta_salida}")
    return df_medidas

In [6]:
# unimos csv de la Comunidad de Madrid en uno solo
df_cmadrid = unir_archivos("../data/raw", "cmadrid_20", "../data/transformed", "cmadrid")

['cmadrid_2005.csv', 'cmadrid_2006.csv', 'cmadrid_2007.csv', 'cmadrid_2008.csv', 'cmadrid_2009.csv', 'cmadrid_2010.csv', 'cmadrid_2011.csv', 'cmadrid_2012.csv', 'cmadrid_2013.csv', 'cmadrid_2014.csv', 'cmadrid_2015.csv', 'cmadrid_2016.csv', 'cmadrid_2017.csv', 'cmadrid_2018.csv', 'cmadrid_2019.csv', 'cmadrid_2020.csv', 'cmadrid_2021.csv', 'cmadrid_2022.csv', 'cmadrid_2023.csv', 'cmadrid_2024.csv', 'cmadrid_2025.csv', 'cmadrid_2025_5.csv']
archivo cmadrid.parquet creado en ../data/transformed


In [7]:
df_cmadrid ["provincia"].unique()

array([28])

In [8]:
# unimos csv de Madrid en uno solo
df_madrid = unir_archivos("../data/raw", "madrid_20", "../data/transformed", "madrid")

['madrid_2001.csv', 'madrid_2002.csv', 'madrid_2003.csv', 'madrid_2004.csv', 'madrid_2005.csv', 'madrid_2006.csv', 'madrid_2007.csv', 'madrid_2008.csv', 'madrid_2009.csv', 'madrid_2010.csv', 'madrid_2011.csv', 'madrid_2012.csv', 'madrid_2013.csv', 'madrid_2014.csv', 'madrid_2015.csv', 'madrid_2016.csv', 'madrid_2017.csv', 'madrid_2018.csv', 'madrid_2019.csv', 'madrid_2020.csv', 'madrid_2021.csv', 'madrid_2022.csv', 'madrid_2023.csv', 'madrid_2024.csv', 'madrid_2025-05-18.csv', 'madrid_2025.csv']
archivo madrid.parquet creado en ../data/transformed


In [9]:
df_madrid ["provincia"].unique()

KeyError: 'provincia'

In [10]:
df_cmadrid[df_cmadrid["h01"] == "22,0"]

,provincia,municipio,estacion,magnitud,punto_muestreo,ano,mes,dia,h01,v01,...,h20,v20,h21,v21,h22,v22,h23,v23,h24,v24
10,28,13,2,7,28013002_7_8,2005,1,1,"22,0",V,...,"68,0",V,"47,0",V,"20,0",V,"8,0",V,"8,0",V
358,28,80,3,8,28080003_8_8,2005,1,3,"22,0",V,...,"55,0",V,"46,0",V,"30,0",V,"23,0",V,"27,0",V
385,28,13,2,14,28013002_14_6,2005,1,4,"22,0",V,...,"14,0",V,"13,0",V,"13,0",V,"12,0",V,"12,0",V
446,28,5,2,8,28005002_8_8,2005,1,4,"22,0",V,...,"167,0",V,"171,0",V,"88,0",V,"52,0",V,"32,0",V
879,28,13,2,8,28013002_8_8,2005,1,8,"22,0",V,...,"54,0",V,"41,0",V,"27,0",V,"22,0",V,"20,0",V
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844811,28,47,2,7,28047002_7_8,2021,12,30,"22,0",V,...,"143,0",V,"98,0",V,"56,0",V,"65,0",V,"73,0",V
844864,28,67,1,12,28067001_12_8,2021,12,30,"22,0",V,...,"75,0",V,"85,0",V,"67,0",V,"47,0",V,"26,0",V
844886,28,9,1,14,28009001_14_6,2021,12,30,"22,0",V,...,"47,0",V,"50,0",V,"52,0",V,"49,0",V,"36,0",V
845007,28,6,4,10,28006004_10_49,2021,12,31,"22,0",V,...,"30,0",V,"44,0",V,"48,0",V,"39,0",V,"33,0",V


In [ ]:
df_cmadrid.columns

Index(['provincia', 'municipio', 'estacion', 'magnitud', 'punto_muestreo',
       'ano', 'mes', 'dia', 'h01', 'v01', 'h02', 'v02', 'h03', 'v03', 'h04',
       'v04', 'h05', 'v05', 'h06', 'v06', 'h07', 'v07', 'h08', 'v08', 'h09',
       'v09', 'h10', 'v10', 'h11', 'v11', 'h12', 'v12', 'h13', 'v13', 'h14',
       'v14', 'h15', 'v15', 'h16', 'v16', 'h17', 'v17', 'h18', 'v18', 'h19',
       'v19', 'h20', 'v20', 'h21', 'v21', 'h22', 'v22', 'h23', 'v23', 'h24',
       'v24'],
      dtype='object')

In [11]:
# Transformamos el df_madrid para que tenga el mismo formato que df_cmadrid
# eliminamos duplicados
df_madrid.drop_duplicates(inplace=True)
# eliminamos una columna con muchos nulos
df_madrid.drop(columns = ["ï»¿PROVINCIA"], inplace=True)
# pasamos todos los encabezdos a minúsculas
df_madrid.columns = df_madrid.columns.str.lower()
# añadimos la columna de provincia al principio
df_madrid.insert(0, "provincia", "28")
df_madrid.tail(15)

,provincia,municipio,estacion,magnitud,punto_muestreo,ano,mes,dia,h01,v01,...,h20,v20,h21,v21,h22,v22,h23,v23,h24,v24
1073727,28,79,8,9,28079008_9_47,2025,4,3,8.0,V,...,4.0,V,4.0,V,5.0,V,7.0,V,7.0,V
1073728,28,79,8,9,28079008_9_47,2025,4,4,4.0,V,...,1.0,V,0.0,V,3.0,V,1.0,V,8.0,V
1073729,28,79,8,9,28079008_9_47,2025,4,5,4.0,V,...,3.0,V,7.0,V,9.0,V,10.0,V,10.0,V
1073730,28,79,8,9,28079008_9_47,2025,4,6,8.0,V,...,2.0,V,7.0,V,7.0,V,9.0,V,11.0,V
1073731,28,79,8,9,28079008_9_47,2025,4,7,8.0,V,...,0.0,V,1.0,V,9.0,V,16.0,V,12.0,V
1073732,28,79,8,9,28079008_9_47,2025,4,8,12.0,V,...,3.0,V,2.0,V,8.0,V,12.0,V,10.0,V
1073733,28,79,8,9,28079008_9_47,2025,4,9,10.0,V,...,5.0,V,7.0,V,6.0,V,9.0,V,5.0,V
1073734,28,79,8,9,28079008_9_47,2025,4,10,16.0,V,...,7.0,V,6.0,V,13.0,V,12.0,V,8.0,V
1073735,28,79,8,9,28079008_9_47,2025,4,11,9.0,V,...,10.0,V,8.0,V,10.0,V,14.0,V,11.0,V
1073736,28,79,8,9,28079008_9_47,2025,4,12,9.0,V,...,12.0,V,6.0,V,6.0,V,17.0,V,4.0,V


In [12]:
df_cmadrid ["provincia"].unique()

array([28])

In [13]:
df_cmadrid.columns.equals(df_madrid.columns)

True

In [12]:
df_madrid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1073755 entries, 0 to 1073754
Data columns (total 56 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   MUNICIPIO       1073755 non-null  int64  
 1   ESTACION        1073755 non-null  int64  
 2   MAGNITUD        1073755 non-null  int64  
 3   PUNTO_MUESTREO  1073755 non-null  object 
 4   ANO             1073755 non-null  int64  
 5   MES             1073755 non-null  int64  
 6   DIA             1073755 non-null  int64  
 7   H01             1073755 non-null  float64
 8   V01             1073755 non-null  object 
 9   H02             1073755 non-null  float64
 10  V02             1073755 non-null  object 
 11  H03             1073755 non-null  float64
 12  V03             1073755 non-null  object 
 13  H04             1073755 non-null  float64
 14  V04             1073755 non-null  object 
 15  H05             1073755 non-null  float64
 16  V05             1073755 non-null  ob

In [14]:
if df_cmadrid.columns.equals(df_madrid.columns):
    df_medidas_cmadrid = formato_df(df_cmadrid)
    df_medidas_madrid = formato_df(df_madrid)

In [15]:
df_medidas_cmadrid

In [9]:
#if df_madrid.columns.equals(df_cmadrid.columns):
    #df_medidas = pd.concat([df_madrid, df_cmadrid], ignore_index=True)
    #df_medidas.info()

In [10]:
df_medidas_cmadrid.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23728289 entries, 10 to 25357127
Data columns (total 13 columns):
 #   Column           Dtype         
---  ------           -----         
 0   provincia        object        
 1   municipio        object        
 2   estacion         object        
 3   magnitud         object        
 4   punto_muestreo   object        
 5   fecha            object        
 6   hora             object        
 7   valor            float64       
 8   fecha_hora_f     datetime64[ns]
 9   validacion       object        
 10  id_medida        object        
 11  codigo_estacion  object        
 12  codigo_tecnica   object        
dtypes: datetime64[ns](1), float64(1), object(11)
memory usage: 2.5+ GB


In [11]:
df_medidas_madrid.head()

,provincia,municipio,estacion,magnitud,punto_muestreo,fecha,hora,valor,fecha_hora_f,validacion,id_medida,codigo_estacion,codigo_tecnica
0,28,079,4,1,28079004_1_38,2001-04-01,01:00,20.0,2001-04-01 01:00:00,V,28079004_1_38_2001-04-01_01:00_V,28079004,38
1,28,079,4,1,28079004_1_38,2001-04-02,01:00,17.0,2001-04-02 01:00:00,V,28079004_1_38_2001-04-02_01:00_V,28079004,38
2,28,079,4,1,28079004_1_38,2001-04-03,01:00,11.0,2001-04-03 01:00:00,V,28079004_1_38_2001-04-03_01:00_V,28079004,38
3,28,079,4,1,28079004_1_38,2001-04-04,01:00,8.0,2001-04-04 01:00:00,V,28079004_1_38_2001-04-04_01:00_V,28079004,38
4,28,079,4,1,28079004_1_38,2001-04-05,01:00,8.0,2001-04-05 01:00:00,V,28079004_1_38_2001-04-05_01:00_V,28079004,38


In [12]:
df_medidas_madrid["valor"].isnull().sum()

np.int64(0)

In [13]:
df_medidas_madrid.duplicated().sum()

np.int64(0)

In [14]:
df_medidas_cmadrid.head()

,provincia,municipio,estacion,magnitud,punto_muestreo,fecha,hora,valor,fecha_hora_f,validacion,id_medida,codigo_estacion,codigo_tecnica
10,28,013,2,7,28013002_7_8,2005-01-01,01:00,22.0,2005-01-01 01:00:00,V,28013002_7_8_2005-01-01_01:00_V,28013002,8
11,28,013,2,8,28013002_8_8,2005-01-01,01:00,23.0,2005-01-01 01:00:00,V,28013002_8_8_2005-01-01_01:00_V,28013002,8
12,28,013,2,10,28013002_10_49,2005-01-01,01:00,51.0,2005-01-01 01:00:00,V,28013002_10_49_2005-01-01_01:00_V,28013002,49
13,28,013,2,14,28013002_14_6,2005-01-01,01:00,13.0,2005-01-01 01:00:00,V,28013002_14_6_2005-01-01_01:00_V,28013002,6
53,28,045,2,6,28045002_6_48,2005-01-01,01:00,3.2,2005-01-01 01:00:00,V,28045002_6_48_2005-01-01_01:00_V,28045002,48


In [15]:
df_medidas_madrid.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25670933 entries, 0 to 25675151
Data columns (total 13 columns):
 #   Column           Dtype         
---  ------           -----         
 0   provincia        object        
 1   municipio        object        
 2   estacion         object        
 3   magnitud         object        
 4   punto_muestreo   object        
 5   fecha            object        
 6   hora             object        
 7   valor            float64       
 8   fecha_hora_f     datetime64[ns]
 9   validacion       object        
 10  id_medida        object        
 11  codigo_estacion  object        
 12  codigo_tecnica   object        
dtypes: datetime64[ns](1), float64(1), object(11)
memory usage: 2.7+ GB


In [16]:
df_medidas_madrid.head()

,provincia,municipio,estacion,magnitud,punto_muestreo,fecha,hora,valor,fecha_hora_f,validacion,id_medida,codigo_estacion,codigo_tecnica
0,28,079,4,1,28079004_1_38,2001-04-01,01:00,20.0,2001-04-01 01:00:00,V,28079004_1_38_2001-04-01_01:00_V,28079004,38
1,28,079,4,1,28079004_1_38,2001-04-02,01:00,17.0,2001-04-02 01:00:00,V,28079004_1_38_2001-04-02_01:00_V,28079004,38
2,28,079,4,1,28079004_1_38,2001-04-03,01:00,11.0,2001-04-03 01:00:00,V,28079004_1_38_2001-04-03_01:00_V,28079004,38
3,28,079,4,1,28079004_1_38,2001-04-04,01:00,8.0,2001-04-04 01:00:00,V,28079004_1_38_2001-04-04_01:00_V,28079004,38
4,28,079,4,1,28079004_1_38,2001-04-05,01:00,8.0,2001-04-05 01:00:00,V,28079004_1_38_2001-04-05_01:00_V,28079004,38


In [2]:
# cargamos los csv de los datos de contaminantes de la Comunidad de Madrid y de Madrid
df_datos_contaminantes_cmadrid = pd.read_csv("../data/raw/datos_contaminantes_cmadrid.csv")
df_datos_contaminantes_madrid = pd.read_csv("../data/raw/datos_contaminantes_madrid.csv")

In [12]:
# de este df vamos a obtener un diccionario con las unidades y su descripcion
df_unidades = df_datos_contaminantes_cmadrid[["UNIDAD", "DESCRIPCIÓN UNIDAD"]].drop_duplicates()
dict_unidades = df_unidades.set_index("UNIDAD")["DESCRIPCIÓN UNIDAD"].to_dict()
dict_unidades

{'µg/m³': 'microgramos por metro cubico',
 'mg/m³': 'miligramos por metro cúbico'}

In [16]:
# hay que corregir las unidades, 'µg/m 3' es lo mismo que 'µg/m3' y luego hay que ponerlo en el mismo formato que el de cmadrid
df_datos_contaminantes_madrid["Unidad"] = df_datos_contaminantes_madrid["Unidad"].str.replace("µg/m 3", "µg/m3")
df_datos_contaminantes_madrid["Unidad"] = df_datos_contaminantes_madrid["Unidad"].str.replace("3", "³")

In [17]:
df_datos_contaminantes_madrid["descripcion_unidad"] = df_datos_contaminantes_madrid["Unidad"].map(dict_unidades)
df_datos_contaminantes_madrid

,Cod.,Mágnitud,Abrevia.,Unidad,Código \nTécnica,Técnica de medida,descripcion_unidad
0,1,Dióxido de Azufre,SO2,µg/m³,38,Fluorescencia ultravioleta,microgramos por metro cubico
1,6,Monóxido de Carbono,CO,mg/m³,48,Absorción infrarroja,miligramos por metro cúbico
2,7,Monóxido de Nitrógeno,NO,µg/m³,8,Quimioluminiscencia,microgramos por metro cubico
3,8,Dióxido de Nitrógeno,NO2,µg/m³,8,Quimioluminiscencia,microgramos por metro cubico
4,9,Partículas < 2.5 µm,PM2.5,µg/m³,47,Microbalanza/Espectrometría*,microgramos por metro cubico
5,10,Partículas < 10 µm,PM10,µg/m³,47,Microbalanza/Espectrometría*,microgramos por metro cubico
6,12,Óxidos de Nitrógeno,NOx,µg/m³,8,Quimioluminiscencia,microgramos por metro cubico
7,14,Ozono,O3,µg/m³,6,Absorción ultravioleta,microgramos por metro cubico
8,20,Tolueno,TOL,µg/m³,59,Cromatografía de gases,microgramos por metro cubico
9,30,Benceno,BEN,µg/m³,59,Cromatografía de gases,microgramos por metro cubico


In [9]:
columnas_madrid_orden = ['Cod.',
 'Mágnitud',
 'Código \nTécnica',
 'Técnica de medida',
 'Unidad',
 "descripcion_unidad"]

In [18]:
df_datos_contaminantes_madrid.drop(columns=["Abrevia."], inplace=True)


In [21]:
df_datos_contaminantes_cmadrid.columns

Index(['CÓDIGO \nMAGNITUD', 'DESCRIPCIÓN MAGNITUD',
       'CÓDIGO \nTÉCNICA \nDE MEDIDA', 'DESCRIPCIÓN TÉCNICA \nDE MEDIDA',
       'UNIDAD', 'DESCRIPCIÓN UNIDAD'],
      dtype='object')

In [ ]:
df_datos_contaminantes_madrid = df_datos_contaminantes_madrid[columnas_madrid_orden]

df_datos_contaminantes_madrid.head()

,Cod.,Mágnitud,Código \nTécnica,Técnica de medida,Unidad,descripcion_unidad
0,1,Dióxido de Azufre,38,Fluorescencia ultravioleta,µg/m³,microgramos por metro cubico
1,6,Monóxido de Carbono,48,Absorción infrarroja,mg/m³,miligramos por metro cúbico
2,7,Monóxido de Nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cubico
3,8,Dióxido de Nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cubico
4,9,Partículas < 2.5 µm,47,Microbalanza/Espectrometría*,µg/m³,microgramos por metro cubico


In [3]:
def formato_contaminantes(df):
    """
    Formatea el DataFrame de contaminantes para que tenga el formato adecuado.

    Args:
        df (pd.DataFrame): DataFrame original de contaminantes.

    Returns:
        pd.DataFrame: DataFrame formateado.
    """
    
    # cambiamos el nombre de las columnas a snake_case y quitamos las tildes
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df.columns = df.columns.map(unidecode)
    df.columns = df.columns.str.replace('\r', '', regex=True).str.strip()
    df.columns = df.columns.str.replace('\n', '', regex=True).str.strip()
    
    return df

In [25]:
df_datos_contaminantes_cmadrid = formato_contaminantes(df_datos_contaminantes_cmadrid)

In [26]:
df_datos_contaminantes_madrid.columns = df_datos_contaminantes_cmadrid.columns
df_datos_contaminantes_madrid

,codigo_magnitud,descripcion_magnitud,codigo_tecnica_de_medida,descripcion_tecnica_de_medida,unidad,descripcion_unidad
0,1,Dióxido de Azufre,38,Fluorescencia ultravioleta,µg/m³,microgramos por metro cubico
1,6,Monóxido de Carbono,48,Absorción infrarroja,mg/m³,miligramos por metro cúbico
2,7,Monóxido de Nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cubico
3,8,Dióxido de Nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cubico
4,9,Partículas < 2.5 µm,47,Microbalanza/Espectrometría*,µg/m³,microgramos por metro cubico
5,10,Partículas < 10 µm,47,Microbalanza/Espectrometría*,µg/m³,microgramos por metro cubico
6,12,Óxidos de Nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cubico
7,14,Ozono,6,Absorción ultravioleta,µg/m³,microgramos por metro cubico
8,20,Tolueno,59,Cromatografía de gases,µg/m³,microgramos por metro cubico
9,30,Benceno,59,Cromatografía de gases,µg/m³,microgramos por metro cubico


In [4]:
def diccionario_unidades(df):
    """
    Crea un diccionario de unidades a partir del DataFrame de contaminantes.

    Args:
        df (pd.DataFrame): DataFrame de contaminantes.

    Returns:
        dict: Diccionario con las unidades y sus descripciones.
    """
    
    # Obtenemos el diccionario de unidades
    df_unidades = df[["UNIDAD", "DESCRIPCIÓN UNIDAD"]].drop_duplicates()
    dict_unidades = df_unidades.set_index("UNIDAD")["DESCRIPCIÓN UNIDAD"].to_dict()
    
    return dict_unidades

In [ ]:
def tablas_contaminantes(df_cmadrid, df_madrid, dict_unidades):
    """
    Procesa y estructura los datos de contaminantes atmosféricos de Madrid y la Comunidad de Madrid.

    Pasos del proceso:

    1. Normalizar la unidad de medida en `df_madrid`, reemplazando "µg/m 3" por "µg/m3" y "3" por "³".
    2. Mapear las unidades con sus descripciones usando `dict_unidades`.
    3. Eliminar la columna innecesaria "Abrevia." de `df_madrid`.
    4. Reordenar las columnas de `df_madrid` para una estructura homogénea.
    5. Aplicar `formato_contaminantes(df_cmadrid)` para preparar la tabla de datos.
    6. Asegurar que `df_madrid` tenga la misma estructura de columnas que `df_cmadrid`.
    7. Concatenar los datos de contaminantes en un único DataFrame `df_datos_contaminantes`.
    8. Convertir las columnas numéricas "codigo_tecnica_de_medida" y "codigo_magnitud" a tipo string.
    9. Extraer las técnicas de medida únicas en `df_tecnicas`.
    10. Limpiar la descripción de las técnicas de medida, eliminando saltos de línea y espacios extra.
    11. Extraer y procesar los contaminantes en `df_contaminantes`, incluyendo "codigo_magnitud", "descripcion_magnitud", "unidad" y "descripcion_unidad".
    12. Eliminar la fila correspondiente a "Ozono Quimioluminiscencia" de `df_contaminantes`.
    13. Eliminar duplicados basados en la columna "codigo_magnitud".
    
    Args:
        df_cmadrid (pd.DataFrame): DataFrame con datos de contaminantes de la Comunidad de Madrid.
        df_madrid (pd.DataFrame): DataFrame con datos de contaminantes de Madrid.
        dict_unidades (dict): Diccionario de unidades y sus descripciones.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: 
            - `df_tecnicas`: DataFrame con técnicas de medida únicas.
            - `df_contaminantes`: DataFrame con contaminantes procesados y estructurados.
    """

    df_madrid["Unidad"] = df_madrid["Unidad"].str.replace("µg/m 3", "µg/m3")
    df_madrid["Unidad"] = df_madrid["Unidad"].str.replace("3", "³")
    df_madrid["descripcion_unidad"] = df_madrid["Unidad"].map(dict_unidades)
    df_madrid.drop(columns=["Abrevia."], inplace=True)
    columnas_madrid_orden = ['Cod.', 'Mágnitud', 'Código \nTécnica', 'Técnica de medida', 'Unidad', "descripcion_unidad"]
    df_madrid = df_madrid[columnas_madrid_orden]
    formato_contaminantes(df_cmadrid)
    df_madrid.columns = df_cmadrid.columns
    df_datos_contaminantes = pd.concat([df_cmadrid, df_madrid], ignore_index=True)
    # cambiamos todas las columnas con números a string porque no operaremos con ellas
    df_datos_contaminantes[["codigo_tecnica_de_medida", "codigo_magnitud"]] = df_datos_contaminantes[["codigo_tecnica_de_medida", "codigo_magnitud"]].astype(str)
    df_tecnicas = df_datos_contaminantes[["codigo_tecnica_de_medida", "descripcion_tecnica_de_medida"]].drop_duplicates()
    df_tecnicas["descripcion_tecnica_de_medida"] = df_tecnicas["descripcion_tecnica_de_medida"].str.replace('\n', '', regex=True).str.strip()
    df_contaminantes = df_datos_contaminantes[['codigo_magnitud', 'descripcion_magnitud', 'unidad', 'descripcion_unidad']]
    df_contaminantes = df_contaminantes.drop(df_contaminantes[df_contaminantes["descripcion_magnitud"] == "Ozono Quimioluminiscencia"].index)
    df_contaminantes = df_contaminantes.drop_duplicates(subset=["codigo_magnitud"])
    return df_tecnicas, df_contaminantes

In [7]:
dict_unidades = diccionario_unidades(df_datos_contaminantes_cmadrid)

In [8]:
df_tecnicas, df_contaminantes = tablas_contaminantes(df_datos_contaminantes_cmadrid, df_datos_contaminantes_madrid, dict_unidades)
df_tecnicas

,codigo_tecnica_de_medida,descripcion_tecnica_de_medida
0,38,Fluorescencia ultravioleta
1,48,Espectrometría infrarroja no dispersiva
2,8,Quimioluminiscencia
4,49,Absorción beta
8,6,Absorción ultravioleta
10,59,Cromatografía de gases
11,7,Absorción de luz
13,2,Ionización llama
17,48,Absorción infrarroja
20,47,Microbalanza/Espectrometría*


In [9]:
df_contaminantes

,codigo_magnitud,descripcion_magnitud,unidad,descripcion_unidad
0,1,Dióxido de azufre,µg/m³,microgramos por metro cúbico
1,6,Monóxido de carbono,mg/m³,miligramos por metro cúbico
2,7,Monóxido de nitrógeno,µg/m³,microgramos por metro cúbico
3,8,Dióxido de nitrógeno,µg/m³,microgramos por metro cúbico
4,9,"Partículas en suspensión < PM2,5",µg/m³,microgramos por metro cubico
5,10,Partículas en suspensión < PM10,µg/m³,microgramos por metro cubico
6,11,Partículas en suspensión < PM1,µg/m³,microgramos por metro cubico
7,12,Óxidos de nitrógeno,µg/m³,microgramos por metro cúbico
8,14,Ozono,µg/m³,microgramos por metro cubico
10,20,Tolueno,µg/m³,microgramos por metro cúbico


In [24]:
# cambiamos el nombre de las columnas a snake_case y quitamos las tildes
df_datos_contaminantes.columns = df_datos_contaminantes.columns.str.lower().str.replace(" ", "_")
df_datos_contaminantes.columns = df_datos_contaminantes.columns.map(unidecode)
# además eliminamos caracteres extraños
df_datos_contaminantes.columns = df_datos_contaminantes.columns.str.replace('\r', '', regex=True).str.strip()
df_datos_contaminantes.columns = df_datos_contaminantes.columns.str.replace('\n', '', regex=True).str.strip()
df_datos_contaminantes.head()

,codigo_magnitud,descripcion_magnitud,codigo_tecnica_de_medida,descripcion_tecnica_de_medida,unidad,descripcion_unidad
0,1,Dióxido de azufre,38,Fluorescencia ultravioleta,µg/m³,microgramos por metro cúbico
1,6,Monóxido de carbono,48,Espectrometría infrarroja no \ndispersiva,mg/m³,miligramos por metro cúbico
2,7,Monóxido de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
3,8,Dióxido de nitrógeno,8,Quimioluminiscencia,µg/m³,microgramos por metro cúbico
4,9,"Partículas en suspensión < PM2,5",49,Absorción beta,µg/m³,microgramos por metro cubico


In [25]:
import pandas as pd

In [26]:
# el siguente paso es construir un df que contenga la información de todas las estaciones de medición
# para eso tenemos que unir el df de Madrid y el de la Comunidad de Madrid, que tienen más o menos las mismas columnas pero no son iguales
# tendremos que hacer algunos cambios para igualar las estructuras de ambos y poder unirlos
# primero cargamos los dos csv
estaciones_cmadrid = pd.read_csv("../data/raw/cmadrid_Red de Calidad del Aire. Estaciones.csv", sep=";", encoding="latin1")
estaciones_madrid = pd.read_csv("../data/raw/madrid_Calidad del aire. Estaciones de control.csv", sep=";")

In [27]:
# a partir de ellos creamos un df con las zonas de medición, para eso cogemos los valores únicos de la columna "zona_calidad_aire_descripcion" de estaciones_cmadrid
df_zonas = pd.DataFrame(estaciones_cmadrid["zona_calidad_aire_descripcion"].unique())
# eliminamos la palabra "Zona" de la columna
df_zonas[0] = df_zonas[0].str.replace("Zona ", "")
# separamos en dos columnas: zona con el número de la zona y descripcion con el nombre de la zona y eliminamos la columnna original
df_zonas[["codigo_zona", "descripcion"]]= df_zonas[0].str.split(" ", n=1,  expand = True)
df_zonas.drop(columns=[0], inplace=True)
# añadimos la fila con la informacion de la zona de Madrid
df_zonas.loc[6] = ["1", "Madrid"]

In [28]:
def dms_to_decimal(dms_str):
    """
    Convierte una coordenada en formato DMS (Grados, Minutos, Segundos) a decimal.

    La función extrae los grados, minutos y segundos de una cadena en formato DMS
    y los convierte a un número en grados decimales, aplicando el signo correspondiente
    si la dirección es 'S' (Sur) o 'W' (Oeste).

    Args:
    
    dms_str : str
        Cadena de texto con la coordenada en formato DMS. 
        Ejemplo: '40°26'46"N', '79°58'56"W'

    Returns:
    
    float
        Coordenada convertida a formato decimal. Los valores en el hemisferio sur y oeste son negativos.

    Ejemplo de uso:
    --------------
    >>> dms_to_decimal("40°26'46\"N")
    40.446111
    >>> dms_to_decimal("79°58'56\"W")
    -79.982222
    """
    partes = dms_str[:-1].replace("°", " ").replace("'", " ").replace('"', "").split()
    grados = float(partes[0])
    minutos = float(partes[1])
    segundos = float(partes[2]) if len(partes) > 2 else 0  # Puede que no incluya segundos
    direccion = dms_str[-1]  # Último carácter indica dirección (N, S, E, W)

    decimal = grados + (minutos / 60) + (segundos / 3600)

    if direccion in ["S", "W"]:  # Hacer negativo si es Sur u Oeste
        decimal *= -1

    return decimal

In [29]:
# empezamos con estaciones_cmadrid
# extraemos el número de zona de la columna "zona_calidad_aire_descripcion" en una nueva columna "codigo_zona"
estaciones_cmadrid["codigo_zona"] = estaciones_cmadrid["zona_calidad_aire_descripcion"].str.extract(r"Zona (\d+)")
# eliminamos la cadena "estacion_" de todas las columnas
estaciones_cmadrid.columns = estaciones_cmadrid.columns.str.replace("estacion_", "")
# las estaciones de Madrid unifican benceno, tolueno y xileno en la columna BTX, por lo que creamos una nueva columna "analizador_BTX" en estaciones_cmadrid
# con el valor correspondiente a las 3 columnas individuales, si no fueran las 3 iguales, nos da un "Fallo"
estaciones_cmadrid["analizador_BTX"] = estaciones_cmadrid.apply(lambda row: row["analizador_TOL"] if row["analizador_TOL"] == row["analizador_BEN"] == row["analizador_XIL"] else "Fallo", axis=1)
# se pueden eliminar las columnas individuales que ahora son redundantes ["analizador_TOL", "analizador_BEN", "analizador_XIL"]
# por dar la misma estructura que estaciones_madrid, duplicamos la columna "municipio" como "nombre_estacion"
estaciones_cmadrid["nombre_estacion"] = estaciones_cmadrid["municipio"]
# y cambiamos "direccion_postal" a "direccion"
estaciones_cmadrid = estaciones_cmadrid.rename(columns={"direccion_postal":"direccion"})
# convertimos "fecha_alta" a datetime
estaciones_cmadrid["fecha_alta"] = pd.to_datetime(estaciones_cmadrid["fecha_alta"])
#estaciones_cmadrid["fecha_alta"] = estaciones_cmadrid["fecha_alta"].apply(lambda x: x.to_pydatetime())

# # como los sistemas de coordenadas no son exactamente los mismos, transformaremos las columnas 
# "coord_longitud" y "coord_latitud" de estaciones_cmadrid al sistema decimal, equivalente a las 
# columnas "LONGITUD" y "LATITUD" de estaciones_madrid


In [30]:
estaciones_cmadrid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   codigo                         28 non-null     int64         
 1   zona_calidad_aire_descripcion  28 non-null     object        
 2   municipio                      28 non-null     object        
 3   fecha_alta                     28 non-null     datetime64[ns]
 4   tipo_area                      28 non-null     object        
 5   tipo_estacion                  28 non-null     object        
 6   subarea_rural                  28 non-null     object        
 7   direccion                      28 non-null     object        
 8   coord_UTM_ETRS89_x             28 non-null     int64         
 9   coord_UTM_ETRS89_y             28 non-null     int64         
 10  coord_longitud                 28 non-null     object        
 11  coord_latitud        

In [31]:
# primero tenemos que cambiar comas por puntos y covertir a str
estaciones_cmadrid[["coord_longitud", "coord_latitud"]] = estaciones_cmadrid[["coord_longitud", "coord_latitud"]].apply(lambda x: x.str.replace(",", "."))
estaciones_cmadrid[["coord_longitud", "coord_latitud"]] = estaciones_cmadrid[["coord_longitud", "coord_latitud"]].astype(str)
# Función para convertir DMS a decimal


estaciones_cmadrid["latitud"] = estaciones_cmadrid["coord_latitud"].apply(dms_to_decimal)
estaciones_cmadrid["longitud"] = estaciones_cmadrid["coord_longitud"].apply(dms_to_decimal)

In [32]:
# y eliminamos columnas innecesarias
estaciones_cmadrid.drop(columns=['analizador_TOL', 'analizador_BEN', 'analizador_XIL', 'zona_calidad_aire_descripcion', 'coord_UTM_ETRS89_x', 'coord_UTM_ETRS89_y', "coord_latitud", "coord_longitud"], inplace=True)


In [33]:
estaciones_cmadrid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   codigo            28 non-null     int64         
 1   municipio         28 non-null     object        
 2   fecha_alta        28 non-null     datetime64[ns]
 3   tipo_area         28 non-null     object        
 4   tipo_estacion     28 non-null     object        
 5   subarea_rural     28 non-null     object        
 6   direccion         28 non-null     object        
 7   altitud           28 non-null     int64         
 8   analizador_NO     28 non-null     object        
 9   analizador_NO2    28 non-null     object        
 10  analizador_PM10   28 non-null     object        
 11  analizador_PM2_5  28 non-null     object        
 12  analizador_PM1    28 non-null     object        
 13  analizador_O3     28 non-null     object        
 14  analizador_O3Q    28 non-nul

In [34]:
# cambiamos también estaciones_madrid
# separamos nom_tipo en tipo area y tipo estacion
estaciones_madrid[["tipo_area", "tipo_estacion"]] = estaciones_madrid["NOM_TIPO"].str.split(" ", expand=True)
# mapeamos los valores de las columnas de contaminantes al mismo formato que los de cmadrid
estaciones_madrid[['NO2', 'SO2', 'CO', 'PM10', 'PM2_5', 'O3', 'BTX']] = estaciones_madrid[['NO2', 'SO2', 'CO', 'PM10', 'PM2_5', 'O3', 'BTX']].map(lambda x: "Si" if x == "X" else "No")
# renombramos las columnas de contaminantes para que tengan el mismo formato que los de cmadrid
estaciones_madrid = estaciones_madrid.rename(columns={col: f"analizador_{col}" for col in ['NO2', 'SO2', 'CO', 'PM10', 'PM2_5', 'O3', 'BTX']})
# convertimos "fecha_alta" a datetime
estaciones_madrid["Fecha alta"] = pd.to_datetime(estaciones_madrid["Fecha alta"], format = "%d/%m/%Y")
#estaciones_madrid["Fecha alta"] = estaciones_madrid["Fecha alta"].apply(lambda x: x.to_pydatetime())
# elegimos el mismo formato de coordenadas que en estaciones_cmadrid y eliminamos columnas sobrantes
estaciones_madrid.drop(columns=["CODIGO_CORTO", "COD_TIPO", "COD_VIA", "VIA_CLASE", "VIA_PAR", "VIA_NOMBRE", "NOM_TIPO", 
                                'COORDENADA_X_ETRS89','COORDENADA_Y_ETRS89', 'LONGITUD_ETRS89', 'LATITUD_ETRS89'], inplace=True)
# pasamos las columnas que lo necesitan a snake case
columnas_cambio = ['CODIGO', 'ESTACION', 'DIRECCION','LONGITUD', 'LATITUD',
       'ALTITUD','Fecha alta']
estaciones_madrid= estaciones_madrid.rename(columns ={col: col.lower().replace(" ", "_") for col in columnas_cambio})
# renombramos columnas
estaciones_madrid = estaciones_madrid.rename(columns={'longitud_etrs89':'coord_longitud','latitud_etrs89':'coord_latitud', "estacion": "nombre_estacion"})
# añadimos columnas que nos faltan y que, al tratarse de Madrid tienen un valor fijo
estaciones_madrid["municipio"] = "Madrid"
estaciones_madrid["subarea_rural"] = "No aplica"
estaciones_madrid["codigo_zona"] = "1"

In [35]:
df_estaciones = pd.concat([estaciones_cmadrid, estaciones_madrid], ignore_index=True)
# capitalizamos la columna tipo_estacion y rellenamos con "Fondo" los valores nulos, ya que son estaciones colocadas en parques y ese parece ser el valor típico para parques
df_estaciones["tipo_estacion"] = df_estaciones["tipo_estacion"].fillna("fondo").str.title()
df_estaciones["altitud"] = pd.to_numeric(df_estaciones["altitud"], errors="coerce").fillna(0).astype(float)
# Rellenamos valores nulos
df_estaciones[["analizador_NO", "analizador_PM1", "analizador_O3Q",
                "analizador_HCT", "analizador_HNM"]] = df_estaciones[["analizador_NO", "analizador_PM1",
                                                                       "analizador_O3Q", "analizador_HCT", "analizador_HNM"]] .fillna("No")
# convertimos todas las columnas de "analizador_" en tipo bool para eso aplicamos una lambda que mapee "Si" por True y "No" por False
analizadores = [col for col in df_estaciones.columns if col.startswith("analizador_")]
df_estaciones[analizadores] = df_estaciones[analizadores].apply(lambda x: x.map({"Si": True, "No": False})).astype(bool)
df_estaciones = df_estaciones.rename(columns={"codigo":"codigo_estacion"})
df_estaciones["codigo_estacion"] = df_estaciones["codigo_estacion"].astype(str)
df_estaciones["codigo_municipio"] = df_estaciones["codigo_estacion"].str[2:5]

In [36]:
df_estaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   codigo_estacion   52 non-null     object        
 1   municipio         52 non-null     object        
 2   fecha_alta        52 non-null     datetime64[ns]
 3   tipo_area         52 non-null     object        
 4   tipo_estacion     52 non-null     object        
 5   subarea_rural     52 non-null     object        
 6   direccion         52 non-null     object        
 7   altitud           52 non-null     float64       
 8   analizador_NO     52 non-null     bool          
 9   analizador_NO2    52 non-null     bool          
 10  analizador_PM10   52 non-null     bool          
 11  analizador_PM2_5  52 non-null     bool          
 12  analizador_PM1    52 non-null     bool          
 13  analizador_O3     52 non-null     bool          
 14  analizador_O3Q    52 non-nul

In [37]:
# Obtener valores únicos
df_municipios = df_estaciones[["codigo_municipio", "municipio"]].drop_duplicates()
df_municipios["codigo_provincia"] = "28"
df_municipios.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29 entries, 0 to 28
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   codigo_municipio  29 non-null     object
 1   municipio         29 non-null     object
 2   codigo_provincia  29 non-null     object
dtypes: object(3)
memory usage: 928.0+ bytes


In [38]:
df_estaciones.drop(columns=["municipio"], inplace=True)

In [39]:
df_estaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   codigo_estacion   52 non-null     object        
 1   fecha_alta        52 non-null     datetime64[ns]
 2   tipo_area         52 non-null     object        
 3   tipo_estacion     52 non-null     object        
 4   subarea_rural     52 non-null     object        
 5   direccion         52 non-null     object        
 6   altitud           52 non-null     float64       
 7   analizador_NO     52 non-null     bool          
 8   analizador_NO2    52 non-null     bool          
 9   analizador_PM10   52 non-null     bool          
 10  analizador_PM2_5  52 non-null     bool          
 11  analizador_PM1    52 non-null     bool          
 12  analizador_O3     52 non-null     bool          
 13  analizador_O3Q    52 non-null     bool          
 14  analizador_CO     52 non-nul

In [10]:
df_contaminantes.to_parquet("../data/transformed/contaminantes.parquet")
df_tecnicas.to_parquet("../data/transformed/tecnicas.parquet")

In [9]:

df_medidas_cmadrid.to_parquet("../data/transformed/medidas_cmadrid.parquet")
df_medidas_madrid.to_parquet("../data/transformed/medidas_madrid.parquet")



In [42]:
df_estaciones.to_parquet("../data/transformed/estaciones.parquet")

In [43]:
df_municipios.to_parquet("../data/transformed/municipios.parquet")
df_zonas.to_parquet("../data/transformed/zonas.parquet")